# 03 — Phase 3a: Offline Model Comparison (Raw Features)

Compares four anomaly/fraud detection approaches on the raw `creditcard.csv`
features: **Isolation Forest**, **Mahalanobis distance** (robust covariance),
**XGBoost** (supervised), and an **autoencoder** (PyTorch, trained on legitimate
transactions only) — plus the IQR rule-based baseline from `01_eda.ipynb`, re-scored
under the same protocol so all five appear on equal footing.

This is Phase 3a specifically: raw V1-V28 + Amount features, no windowed features
from Phase 2. Phase 3b (retraining the winning approach(es) on Phase 2's windowed
features, joined against `ground_truth.jsonl`) is a separate follow-up.

**Produces:** trained models saved to `models/<name>/<date>/` at the project root
(git-ignored).

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from fraud_pipeline.ingestion.loader import load_transactions
from fraud_pipeline.models.autoencoder import fit_autoencoder, save_autoencoder_artifact
from fraud_pipeline.models.autoencoder import score as score_autoencoder
from fraud_pipeline.models.evaluation import evaluate, save_model_artifact, threshold_for_target_fpr
from fraud_pipeline.models.isolation_forest import fit_isolation_forest
from fraud_pipeline.models.isolation_forest import score as score_isolation_forest
from fraud_pipeline.models.mahalanobis import fit_mahalanobis
from fraud_pipeline.models.mahalanobis import score as score_mahalanobis
from fraud_pipeline.models.xgboost_baseline import fit_xgboost
from fraud_pipeline.models.xgboost_baseline import score as score_xgboost

df = load_transactions()
print(f"shape: {df.shape}")

shape: (284807, 31)


## Temporal train/test split

Split on `Time` at the 70th percentile, computed from the data rather than
hard-coded — training data is always chronologically before test data, closer to how
this system would actually be evaluated (train on the past, test on more recent
transactions) than a random shuffle would be.

In [2]:
feature_cols = [c for c in df.columns if c.startswith("V")] + ["Amount"]
print(f"{len(feature_cols)} features: {feature_cols}")

time_cutoff = df["Time"].quantile(0.70)
train_df = df[df["Time"] <= time_cutoff]
test_df = df[df["Time"] > time_cutoff]

print(f"\ncutoff (Time): {time_cutoff:.1f}")
print(f"train: {len(train_df):,} rows, {train_df['Class'].mean():.4%} fraud")
print(f"test:  {len(test_df):,} rows, {test_df['Class'].mean():.4%} fraud")

29 features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

cutoff (Time): 132929.0
train: 199,368 rows, 0.1926% fraud
test:  85,439 rows, 0.1264% fraud


In [3]:
X_train = train_df[feature_cols].values
y_train = train_df["Class"].values
X_test = test_df[feature_cols].values
y_test = test_df["Class"].values

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (199368, 29), X_test: (85439, 29)


## Baseline: re-scoring the IQR rule under this protocol

`01_eda.ipynb`'s IQR rule (91.7% recall / 16.0% FPR) was computed on the *full*
dataset with no train/test split — useful for EDA, but not directly comparable to
models evaluated on a held-out test set. Re-deriving it here under the same
discipline as the three new models: rank features by Cohen's d **on the training
split only**, compute IQR bounds **on the training split only**, then flag the test
split. This avoids the same leakage train/test splitting is meant to prevent
elsewhere in this project, and makes the number that follows genuinely comparable to
the other three.

In [4]:
def cohens_d(a, b):
    pooled_std = ((a.std() ** 2 + b.std() ** 2) / 2) ** 0.5
    return (a.mean() - b.mean()) / pooled_std


fraud_train = train_df[train_df["Class"] == 1]
normal_train = train_df[train_df["Class"] == 0]

separation = pd.Series(
    {col: cohens_d(fraud_train[col], normal_train[col]) for col in feature_cols}
).sort_values(key=abs, ascending=False)

iqr_features = separation.abs().sort_values(ascending=False).index[:6].tolist()
print(f"top 6 features by |Cohen's d| on train: {iqr_features}")

top 6 features by |Cohen's d| on train: ['V14', 'V4', 'V12', 'V11', 'V10', 'V16']


In [5]:
def iqr_outlier_mask(series, bounds, k=1.5):
    q1, q3 = bounds
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)


iqr_bounds = {col: train_df[col].quantile([0.25, 0.75]).tolist() for col in iqr_features}

test_flags = pd.DataFrame(
    {col: iqr_outlier_mask(test_df[col], iqr_bounds[col]) for col in iqr_features}
)
iqr_pred = test_flags.any(axis=1).astype(int).values

iqr_metrics = evaluate(y_test, y_pred=iqr_pred, y_score=iqr_pred.astype(float))
iqr_metrics

{'pr_auc': 0.007372896572134922,
 'roc_auc': 0.8857454110073323,
 'recall': 0.9166666666666666,
 'precision': 0.007928245375190197,
 'false_positive_rate': 0.1451758446520022}

(The IQR rule's `pr_auc`/`roc_auc` above are degenerate — a fixed either/or rule has
only one real operating point, not a tunable score, so its "curve" is really just two
points. They're included so the comparison table has a value in every column, not
because they mean the same thing as the other models' curve AUCs.)

## Isolation Forest

Fit on the training split **as-is** — fraud included, not filtered out — matching how
this would actually be deployed: a real unsupervised detector never gets to see
`Class` to clean its own training data.

In [6]:
iso_forest = fit_isolation_forest(X_train, random_state=42)
iso_scores = score_isolation_forest(iso_forest, X_test)
print(f"score range: [{iso_scores.min():.3f}, {iso_scores.max():.3f}]")

score range: [-0.157, 0.284]


## Mahalanobis distance (robust covariance)

`MinCovDet`, not a plain covariance matrix — a plain estimate gets pulled toward the
fraud already present in the training data, weakening what "normal" even means.
Features are standardized first (see `mahalanobis.py` for why).

In [7]:
scaler, mcd = fit_mahalanobis(X_train, random_state=42)
mahalanobis_scores = score_mahalanobis(scaler, mcd, X_test)
print(f"score range: [{mahalanobis_scores.min():.3f}, {mahalanobis_scores.max():.3f}]")

score range: [7.319, 4371104479.282]


**A real finding, not swept under the rug.** That max score is enormous -- billions,
not the low hundreds a chi-square(29) distribution would suggest. The cause: the
robust covariance's smallest eigenvalue is ~8.5e-8, a near-singular direction giving a
condition number around 15 million. That inflates distances for roughly 2% of the
test set (1,725 of 85,439 points score above 1e6), not just one or two outliers.

The standard fix for a near-singular covariance is ridge/shrinkage regularization
before inverting. Tested at several shrinkage levels against this exact split:
ROC-AUC barely moves (0.9489 -> 0.9480 at shrinkage=0.05), but **PR-AUC drops
sharply** (0.53 -> 0.07) -- the near-singular direction is doing real work
concentrating true fraud cases near the top of the ranking, not just adding noise.
At the matched-FPR operating point, regularization did nudge recall up slightly
(86.1% -> 89.8%), so the tradeoff genuinely cuts both ways depending which metric
matters more.

Given the spec calls for `MinCovDet` specifically with no mention of additional
shrinkage, and there's no clean win either way, the unregularized fit ships as the
result below -- but this is exactly the kind of fragility worth watching if this
model is ever retrained on different data (Phase 3b's windowed features, for
instance) rather than assuming it will stay this well-behaved by accident.

## XGBoost (supervised)

The one model here that trains on `Class` directly — a ceiling, not a like-for-like
comparison with the two unsupervised methods above.

In [8]:
xgb_model = fit_xgboost(X_train, y_train, random_state=42)
xgb_scores = score_xgboost(xgb_model, X_test)
print(f"score range: [{xgb_scores.min():.3f}, {xgb_scores.max():.3f}]")

score range: [0.000, 1.000]


## Autoencoder

The odd one out among the unsupervised methods: Isolation Forest and Mahalanobis were
both fit on the training split *as-is*, fraud included — Isolation Forest assumes
anomalies are rare and doesn't need them excluded, and `MinCovDet` is specifically a
robust estimator built to resist contamination. A neural network has no such built-in
safeguard: given enough capacity and training time, it can learn to reconstruct fraud
just as well as legitimate transactions, which would quietly defeat the entire premise
of using reconstruction error as an anomaly score. So this one trains on **legitimate
transactions only**.

(A real deadlock was found and fixed while building this: XGBoost and PyTorch each
load their own OpenMP runtime, and on macOS having both loaded in one process hangs
indefinitely the moment PyTorch runs its first parallel op after XGBoost has already
claimed the thread pool — `fit_xgboost()` above completed fine, then `fit_autoencoder()`
hung with no error at all. Fixed in `autoencoder.py` itself, not here, since importing
it is what introduces the second OpenMP runtime in the first place.)

In [9]:
train_legit_df = train_df[train_df["Class"] == 0].sort_values("Time")
X_train_legit = train_legit_df[feature_cols].values

n_dropped = len(train_df) - len(X_train_legit)
print(
    f"legitimate-only training rows: {len(X_train_legit):,} "
    f"(dropped {n_dropped} fraud rows -- this model's training data only)"
)

legitimate-only training rows: 198,984 (dropped 384 fraud rows -- this model's training data only)


A validation slice for early stopping is carved from the *end* of this legitimate-only
training data (most recent ~15%, preserving temporal order) — not from the real test
split, which stays completely untouched until scoring below.

In [10]:
ae_scaler, ae_model, ae_info = fit_autoencoder(X_train_legit, random_state=42)

print(f"architecture: {ae_model.architecture_description()}")
print(f"epochs trained: {ae_info['epochs_trained']} (early stopping, patience=10)")
print(f"final validation loss: {ae_info['final_val_loss']:.6f}")

architecture: 29 -> 20 -> 14 -> 7 -> 14 -> 20 -> 29
epochs trained: 65 (early stopping, patience=10)
final validation loss: 0.577032


In [11]:
autoencoder_scores = score_autoencoder(ae_scaler, ae_model, X_test)
print(f"score range: [{autoencoder_scores.min():.3f}, {autoencoder_scores.max():.3f}]")

score range: [0.033, 265.827]


## Comparison table

Two things happening here, not one:

- **Matched operating point**: Isolation Forest, Mahalanobis, XGBoost, and the
  autoencoder each get a threshold chosen so their false-positive rate matches the IQR
  baseline's — makes recall comparable at a fixed false-positive "budget" instead of
  comparing across four different, arbitrarily-chosen operating points.
- **Full-curve AUCs** (`pr_auc`, `roc_auc`): threshold-independent, computed once per
  model regardless of the matched point above.

One honest caveat: the matched threshold is chosen using the *test set's* labels,
purely to build this comparison table. That's a reasonable simplification for a
reporting table but not something to deploy as-is — a real deployment threshold
should come from a separate validation split, not the same data used to report
results.

In [12]:
target_fpr = iqr_metrics["false_positive_rate"]
print(f"target FPR (from IQR baseline): {target_fpr:.4f}")

results = {"IQR baseline": iqr_metrics}

for name, scores in [
    ("Isolation Forest", iso_scores),
    ("Mahalanobis (MinCovDet)", mahalanobis_scores),
    ("XGBoost", xgb_scores),
    ("Autoencoder", autoencoder_scores),
]:
    threshold = threshold_for_target_fpr(y_test, scores, target_fpr)
    y_pred = (scores >= threshold).astype(int)
    results[name] = evaluate(y_test, y_pred=y_pred, y_score=scores)

comparison = pd.DataFrame(results).T[
    ["recall", "false_positive_rate", "precision", "pr_auc", "roc_auc"]
]
comparison.round(4)

target FPR (from IQR baseline): 0.1452


,recall,false_positive_rate,precision,pr_auc,roc_auc
IQR baseline,0.9167,0.1452,0.0079,0.0074,0.8857
Isolation Forest,0.8889,0.1455,0.0077,0.0396,0.9456
Mahalanobis (MinCovDet),0.8611,0.1449,0.0075,0.5327,0.9489
XGBoost,0.9352,0.1451,0.0081,0.7988,0.9825
Autoencoder,0.8704,0.1429,0.0077,0.1063,0.9372


## Save trained models

`data/raw/creditcard.csv`'s labels are never used by the two unsupervised models at
prediction time — only the trained artifact and this evaluation notebook use `Class`,
and only for training XGBoost or scoring afterward, matching Phase 1's no-leakage
discipline about what a deployed model can and can't see.

In [13]:
save_model_artifact(
    iso_forest,
    name="isolation_forest",
    hyperparameters=iso_forest.get_params(),
    metrics=results["Isolation Forest"],
)

save_model_artifact(
    {"scaler": scaler, "model": mcd},
    name="mahalanobis",
    hyperparameters={"covariance": mcd.get_params(), "scaler": "StandardScaler"},
    metrics=results["Mahalanobis (MinCovDet)"],
)

save_model_artifact(
    xgb_model,
    name="xgboost",
    hyperparameters=xgb_model.get_params(),
    metrics=results["XGBoost"],
)

save_autoencoder_artifact(
    ae_model,
    ae_scaler,
    hyperparameters={
        **ae_info["hyperparameters"],
        "epochs_trained": ae_info["epochs_trained"],
        "final_val_loss": ae_info["final_val_loss"],
    },
    metrics=results["Autoencoder"],
)

print("saved 4 model artifacts to models/")

saved 4 model artifacts to models/


## Key Takeaways

- Re-scoring the IQR baseline under a proper train/test protocol (rather than reusing
  the full-dataset number from `01_eda.ipynb`) is what makes this comparison fair —
  see the actual re-scored numbers in the table above.
- Isolation Forest and Mahalanobis were both fit on training data that still contains
  fraud (~0.17-0.2%, matching real deployment conditions) — no cheating by filtering
  the training set first. The autoencoder is the one exception: it trains on
  legitimate transactions only, because unlike the other two it has no built-in
  robustness to contamination.
- The robust covariance estimate (`MinCovDet`) is specifically meant to resist being
  pulled toward that small fraction of fraud in the training data, unlike a plain
  covariance matrix.
- XGBoost is trained with `Class` directly and represents a ceiling, not a fair
  comparison with the three unsupervised methods — expect it to outperform them, and
  don't read that as the unsupervised approaches being weak.
- The matched-FPR comparison isolates *which model finds more fraud at the same
  false-positive cost* — see the table above for this run's actual numbers.
- **All four methods originally planned for Phase 3a are now built and compared** —
  this notebook can be considered Phase 3a's complete deliverable once the five-row
  table above has been generated and reviewed.
- Phase 3b is next: whichever approach(es) look most promising here get retrained on
  Phase 2's windowed features, which requires building a join against
  `ground_truth.jsonl` and deciding a window-level fraud label rule (e.g. "any
  transaction in this window is fraud" vs. a stricter rule) — neither exists yet.